---
# Notebook to prep direct images to work with wfss extraction

This notebook takes mosaics downloaded from DJA and prepares them to work with the wfss extraction notebook. If you are using mosaics from another source you may need to alter some lines to account for differences in image headers.

---

## 00. Import modules and identify the avaiable mosaics.

In [14]:
import os
import glob
import re
import warnings

WORK_ROOT = os.path.abspath("/data/users/cooper/WFSS_GRIZLI_NEW/")   # same as the main notebook


import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table
from astropy.wcs import WCS

import grizli
from grizli import utils, prep

utils.set_warnings()
warnings.filterwarnings("ignore", category=RuntimeWarning)

FIELDS = ["a2744"]

MOSAICS = {}
for field in FIELDS:
    MOSAICS[field] = {}
    for sci in sorted(glob.glob(os.path.join(WORK_ROOT,  "dir_imgs", field, "*_sci.fits*"))): # Define the directory to your mosaics here
        filt = re.search(r"-(f\d{3}[wmn]2?)-clear_dr[cz]_sci", os.path.basename(sci)).group(1)
        wht = sci.replace("_sci.fits", "_wht.fits")
        if not os.path.exists(wht):
            print(f"WARNING: no weight image for {os.path.basename(sci)}")
            continue
        MOSAICS[field][filt] = (sci, wht)

    filts = sorted(MOSAICS[field])
    print(f"{field:>6}: {len(filts)} filters  {filts if filts else '(none found)'}")

 a2744: 9 filters  ['f277w', 'f300m', 'f335m', 'f356w', 'f360m', 'f410m', 'f444w', 'f460m', 'f480m']


---
## 01. Build the long wavelength detection image

Grizli needs a segmentation image and catalog to model contamination. I build these using a stacked long wavelength, wide band, image produced by the cell below. Files are saved following grizli's conventions.

---


In [15]:
DET_FILTERS = ["f277w", "f356w", "f444w"]

for field in FIELDS:
    out_root = os.path.join(WORK_ROOT, field, "Prep", f"{field}-ir")
    if os.path.exists(f"{out_root}_drz_sci.fits"):
        print(f"{field}: detection image exists, skipping")
        continue

    use = [f for f in DET_FILTERS if f in MOSAICS[field]]
    if not use:
        print(f"{field}: none of {DET_FILTERS} available, skipping")
        continue
    print(f"{field}: stacking {use}")

    num = den = ref_header = None
    for filt in use:
        sci_file, wht_file = MOSAICS[field][filt]
        with fits.open(sci_file) as hs, fits.open(wht_file) as hw:
            sci = hs[0].data.astype(np.float32)
            wht = hw[0].data.astype(np.float32)
            if ref_header is None:
                ref_header = hs[0].header
                num = np.zeros_like(sci)
                den = np.zeros_like(sci)
            elif sci.shape != num.shape:
                raise ValueError(f"{filt} grid {sci.shape} differs from {num.shape}")

        good = np.isfinite(sci) & np.isfinite(wht) & (wht > 0)
        num[good] += sci[good] * wht[good]
        den[good] += wht[good]
        del sci, wht, good

    det = np.zeros_like(num)
    np.divide(num, den, out=det, where=den > 0)
    del num

    hdr = WCS(ref_header).to_header()
    hdr["PHOTFNU"] = (1e-8, "Jy per pixel unit (10 nJy)")
    hdr["FILTER"] = ("ir", "Detection stack")
    hdr["DETFILTS"] = (",".join(use), "Filters in detection stack")

    fits.writeto(f"{out_root}_drz_sci.fits", det, hdr, overwrite=True)
    fits.writeto(f"{out_root}_drz_wht.fits", den, hdr, overwrite=True)
    print(f"  -> {out_root}_drz_sci.fits  {det.shape}")
    del det, den

a2744: stacking ['f277w', 'f356w', 'f444w']
  -> /data/users/cooper/WFSS_GRIZLI_NEW/a2744/Prep/a2744-ir_drz_sci.fits  (16384, 21504)


### If you are working with large direct images but small FoV GRISM images, this cell crops the LW stack to match the overall GRISM footprint.

In [16]:
PAD_ARCSEC = 60     # Margin to account for off-sensor dispersion

for field in FIELDS:
    prep_dir = os.path.join(WORK_ROOT, field, "Prep")
    root = os.path.join(prep_dir, f"{field}-ir")
    full = os.path.join(prep_dir, f"{field}-ir-full")
    if os.path.exists(f"{full}_drz_sci.fits"):
        print(f"{field}: already cropped, skipping")
        continue
    if not os.path.exists(f"{root}_drz_sci.fits"):
        print(f"{field}: no detection image, skipping")
        continue

    # Grism footprint corners from the initialised grism exposures
    corners = []
    for f in glob.glob(os.path.join(WORK_ROOT, field, "RAW", "*_rate.fits")):
        if "GRISM" in fits.getval(f, "PUPIL", ext=0):
            corners.append(WCS(fits.getheader(f, "SCI")).calc_footprint())
    corners = np.vstack(corners)

    # Pixel box in the detection image
    with fits.open(f"{root}_drz_sci.fits", memmap=True) as h:
        hdr, ny, nx = h[0].header, *h[0].data.shape
    wcs = WCS(hdr)
    xs, ys = wcs.world_to_pixel_values(corners[:, 0], corners[:, 1])
    pad = PAD_ARCSEC / 0.04
    x0, x1 = int(max(xs.min() - pad, 0)), int(min(xs.max() + pad, nx))
    y0, y1 = int(max(ys.min() - pad, 0)), int(min(ys.max() + pad, ny))
    print(f"{field}: {len(corners) // 4} grism exposures, crop {x1 - x0} x {y1 - y0} "
          f"of {nx} x {ny} pixels")

    # Keep full images under a new name, write the crop under the original name
    for ext in ["sci", "wht"]:
        os.rename(f"{root}_drz_{ext}.fits", f"{full}_drz_{ext}.fits")

    crop_hdr = wcs[y0:y1, x0:x1].to_header()
    for k in ["PHOTFNU", "FILTER", "DETFILTS"]:
        crop_hdr[k] = hdr[k]
    crop_hdr["CROPX0"] = (x0, "x offset of crop in full DJA mosaic (0-based)")
    crop_hdr["CROPY0"] = (y0, "y offset of crop in full DJA mosaic (0-based)")

    for ext in ["sci", "wht"]:
        with fits.open(f"{full}_drz_{ext}.fits", memmap=True) as h:
            fits.writeto(f"{root}_drz_{ext}.fits", np.array(h[0].data[y0:y1, x0:x1]),
                         crop_hdr, overwrite=True)

a2744: 60 grism exposures, crop 13115 x 12858 of 21504 x 16384 pixels


### Produce the source catalogue and segmentation file using grizli's built in SEP tools
Thesse SEP parameters have been decided for crowded fields (lensing clusters), so it may be worth experimenting with 

In [17]:
from grizli.prep import SEP_DETECT_PARAMS

THRESHOLD = 2.5
MESH = 32
DEBLEND_CONT = 0.0003
OVERWRITE = True

for field in FIELDS:
    prep_dir = os.path.join(WORK_ROOT, field, "Prep")
    root = f"{field}-ir"
    if not os.path.exists(os.path.join(prep_dir, f"{root}_drz_sci.fits")):
        print(f"{field}: no detection image, skipping")
        continue

    os.chdir(prep_dir)
    if OVERWRITE:
        for f in [f"{root}.cat.fits", f"{root}_seg.fits", f"{root}_bkg.fits"]:
            if os.path.exists(f):
                os.remove(f)
    elif os.path.exists(f"{root}.cat.fits"):
        print(f"{field}: catalogue exists, skipping")
        continue

    cat = prep.make_SEP_catalog(
        root=root,
        threshold=THRESHOLD,
        get_background=True,
        bkg_params={"bw": MESH, "bh": MESH, "fw": 3, "fh": 3},
        detection_params={**SEP_DETECT_PARAMS, "deblend_cont": DEBLEND_CONT},
        pixel_scale=0.04,
        extract_pixstack=int(3e7),
        verbose=False,
    )
    print(f"{field}: {len(cat)} sources -> {root}.cat.fits, {root}_seg.fits")

os.chdir(WORK_ROOT)

a2744: 34145 sources -> a2744-ir.cat.fits, a2744-ir_seg.fits


### Crop each direct image to match the GRISM footprint

In [18]:
for field in FIELDS:
    prep_dir = os.path.join(WORK_ROOT, field, "Prep")
    det_file = os.path.join(prep_dir, f"{field}-ir_drz_sci.fits")
    full_file = os.path.join(prep_dir, f"{field}-ir-full_drz_sci.fits")
    if not os.path.exists(det_file):
        print(f"{field}: no detection image, skipping")
        continue

    det_hdr = fits.getheader(det_file)
    x0, y0 = det_hdr["CROPX0"], det_hdr["CROPY0"]
    ny, nx = det_hdr["NAXIS2"], det_hdr["NAXIS1"]
    full_shape = fits.getdata(full_file, memmap=True).shape
    print(f"{field}: crop [{y0}:{y0 + ny}, {x0}:{x0 + nx}] of {full_shape}")

    for filt, (sci_file, wht_file) in MOSAICS[field].items():
        out = os.path.join(prep_dir, f"{field}-{filt}-clear_drz")
        if os.path.exists(f"{out}_sci.fits"):
            print(f"  exists   {filt}")
            continue

        for ext, src in [("sci", sci_file), ("wht", wht_file)]:
            with fits.open(src) as h:
                data, hdr = h[0].data, h[0].header.copy()
                if data.shape != full_shape:
                    raise ValueError(f"{os.path.basename(src)} grid {data.shape} "
                                     f"differs from detection grid {full_shape}")
                crop = np.array(data[y0:y0 + ny, x0:x0 + nx], dtype=np.float32)
            hdr["CRPIX1"] -= x0
            hdr["CRPIX2"] -= y0
            hdr["CROPX0"] = (x0, "x offset of crop in full DJA mosaic (0-based)")
            hdr["CROPY0"] = (y0, "y offset of crop in full DJA mosaic (0-based)")
            fits.writeto(f"{out}_{ext}.fits", crop, hdr, overwrite=True)
        print(f"  cropped  {filt}")

a2744: crop [606:13464, 2366:15481] of (16384, 21504)
  cropped  f277w
  cropped  f300m
  cropped  f335m
  cropped  f356w
  cropped  f360m
  cropped  f410m
  cropped  f444w
  cropped  f460m
  cropped  f480m
